# GOES ABI-L2-MCMIPC preview

Preview the downloaded GOES imagery in `/mnt/disk1/goes-data/`. Pick a date, then either take
a quick **static** look at a band / true-color RGB, or drop it onto an
**interactive map** you can scroll-zoom and pan — the image is reprojected to
lat/lon and overlaid on a basemap (OSM / light / satellite).

Every file holds all 16 ABI bands:

| # | µm | type | # | µm | type |
|---|----|------|---|----|------|
| 1 | 0.47 blue | reflectance | 9 | 6.9 mid water-vapor | brightness temp |
| 2 | 0.64 red | reflectance | 10 | 7.3 low water-vapor | brightness temp |
| 3 | 0.86 veggie | reflectance | 11 | 8.4 cloud-top phase | brightness temp |
| 4 | 1.37 cirrus | reflectance | 12 | 9.6 ozone | brightness temp |
| 5 | 1.6 snow/ice | reflectance | 13 | 10.3 clean IR | brightness temp |
| 6 | 2.2 cloud particle | reflectance | 14 | 11.2 IR | brightness temp |
| 7 | 3.9 shortwave IR | brightness temp | 15 | 12.3 dirty IR | brightness temp |
| 8 | 6.2 upper water-vapor | brightness temp | 16 | 13.3 CO₂ | brightness temp |

True color ≈ RGB from bands **(2, 3, 1)**.

## Helpers — run this cell once

In [ ]:
import base64
import io
from datetime import date
from pathlib import Path

import folium
import matplotlib.pyplot as plt
import numpy as np
import pyproj
import rioxarray  # noqa: F401  (registers the .rio accessor)
import xarray as xr
from folium.raster_layers import ImageOverlay
from PIL import Image

# GOES imagery lives on the data drive (override per call with data_dir=)
DATA_DIR = Path("/mnt/disk1/goes-data")


# ---- file discovery & selection ----
def _scan_token(p):
    for part in p.name.split("_"):
        if part.startswith("s") and part[1:].isdigit():
            return part
    return p.name


def scan_time(p):
    t = _scan_token(p)
    if t.startswith("s") and len(t) >= 12:
        return f"{t[8:10]}:{t[10:12]} UTC"
    return "??:??"


def find_files(dt, data_dir=DATA_DIR):
    pat = f"*/{dt.year}/{dt.month:02d}/{dt.day:02d}/*.nc"
    return sorted(data_dir.glob(pat), key=_scan_token)


def open_goes(dt, file_index=0, data_dir=DATA_DIR):
    """List the files for a date and open one (default the first/only)."""
    files = find_files(dt, data_dir)
    if not files:
        raise FileNotFoundError(f"No .nc files for {dt} under {data_dir}")
    for i, p in enumerate(files):
        arrow = "->" if i == file_index else "  "
        print(f"{arrow} [{i}] {scan_time(p)}  {p.name}")
    return xr.open_dataset(files[file_index], decode_times=False)


# ---- band access & scaling ----
def cmi(ds, n):
    """The CMI_C<n> DataArray (decoded reflectance or brightness temperature)."""
    return ds[f"CMI_C{n:02d}"]


def _is_bt(da):
    return str(da.attrs.get("units", "")).strip().upper().startswith("K")


def _cmap(da):
    return "gray_r" if _is_bt(da) else "gray"  # IR: cold cloud tops = white


def _stretch(a, vmin=None, vmax=None):
    lo = np.nanpercentile(a, 2) if vmin is None else vmin
    hi = np.nanpercentile(a, 98) if vmax is None else vmax
    return float(lo), float(hi)


def _norm(a, lo, hi):
    return np.clip((a - lo) / (hi - lo), 0, 1) if hi > lo else np.zeros_like(a)


# ---- crop on the native geostationary grid ----
def _geos(ds):
    return pyproj.CRS.from_cf(dict(ds["goes_imager_projection"].attrs))


def crop_lonlat(ds, lon_min, lon_max, lat_min, lat_max):
    """Subset to a lon/lat box (degrees) before plotting/mapping."""
    h = ds["goes_imager_projection"].attrs["perspective_point_height"]
    tf = pyproj.Transformer.from_crs("EPSG:4326", _geos(ds), always_xy=True)
    lons = [lon_min, lon_max, lon_min, lon_max]
    lats = [lat_min, lat_min, lat_max, lat_max]
    xs, ys = tf.transform(lons, lats)
    xs = np.array(xs) / h
    ys = np.array(ys) / h
    xs, ys = xs[np.isfinite(xs)], ys[np.isfinite(ys)]
    if xs.size == 0 or ys.size == 0:
        raise ValueError("box is off the Earth disk for this satellite")
    return ds.sel(x=slice(xs.min(), xs.max()), y=slice(ys.max(), ys.min()))


def crop_pixels(ds, x0, x1, y0, y1):
    """Subset by pixel index (x: 0..2500 W->E, y: 0..1500 N->S)."""
    return ds.isel(x=slice(x0, x1), y=slice(y0, y1))


# ---- static (matplotlib) previews ----
def show_band(ds, n, cmap=None, vmin=None, vmax=None, figsize=(11, 7)):
    da = cmi(ds, n)
    a = da.values
    lo, hi = _stretch(a, vmin, vmax)
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(a, cmap=cmap or _cmap(da), vmin=lo, vmax=hi,
                   interpolation="nearest", origin="upper", aspect="equal")
    units = da.attrs.get("units", "")
    ax.set_title(f"Band {n}: {da.attrs.get('long_name', '')} ({units})\n"
                 f"{a.shape[1]}x{a.shape[0]} px")
    fig.colorbar(im, ax=ax, shrink=0.7, label=units)
    fig.tight_layout()
    return fig


def show_rgb(ds, bands=(2, 3, 1), gamma=2.2, figsize=(11, 7)):
    chans = []
    for b in bands:
        a = cmi(ds, b).values
        chans.append(_norm(a, *_stretch(a)))
    rgb = np.clip(np.nan_to_num(np.dstack(chans), nan=0.0) ** (1 / gamma), 0, 1)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(rgb, interpolation="nearest", origin="upper", aspect="equal")
    ax.set_title(f"RGB = bands {tuple(bands)} (R, G, B)\n"
                 f"{rgb.shape[1]}x{rgb.shape[0]} px")
    fig.tight_layout()
    return fig


# ---- reproject geostationary -> lat/lon, then RGBA for a web-map overlay ----
def reproject_bands(ds, bands):
    """Reproject bands to EPSG:4326.

    Returns (DataArray with dim 'band', bounds=[[south, west], [north, east]]),
    oriented north-up / west-left so it drops straight onto a Leaflet overlay.
    """
    h = ds["goes_imager_projection"].attrs["perspective_point_height"]
    da = xr.concat([cmi(ds, n).reset_coords(drop=True) for n in bands], dim="band")
    da = da.assign_coords(x=ds["x"] * h, y=ds["y"] * h)
    da = da.rio.write_crs(_geos(ds)).rio.set_spatial_dims(x_dim="x", y_dim="y")
    da = da.rio.reproject("EPSG:4326", nodata=np.nan)
    da = da.sortby("y", ascending=False).sortby("x")
    minx, miny, maxx, maxy = da.rio.bounds()
    return da, [[miny, minx], [maxy, maxx]]


def _rgba_single(a, cmap, lo, hi):
    finite = np.isfinite(a)
    sm = plt.cm.ScalarMappable(norm=plt.Normalize(lo, hi), cmap=cmap)
    rgba = sm.to_rgba(np.nan_to_num(a, nan=lo), bytes=True)
    rgba[..., 3] = np.where(finite, 255, 0).astype("uint8")
    return rgba


def _rgba_rgb(arr3, gamma):
    chans, alpha = [], np.ones(arr3.shape[1:], dtype=bool)
    for a in arr3:
        chans.append(_norm(a, *_stretch(a)))
        alpha = alpha & np.isfinite(a)
    rgb = np.nan_to_num(np.clip(np.dstack(chans) ** (1 / gamma), 0, 1))
    a8 = np.where(alpha, 255, 0).astype("uint8")
    return np.dstack([(rgb * 255).astype("uint8"), a8])


def _data_uri(rgba):
    buf = io.BytesIO()
    Image.fromarray(rgba, mode="RGBA").save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


def show_on_map(ds, band=13, rgb=None, cmap=None, vmin=None, vmax=None,
                gamma=2.2, opacity=0.85):
    """Overlay a GOES band (or RGB triple) on a scroll-zoom folium map.

    The image is reprojected to lat/lon so it aligns with the basemap. Use the
    layer control (top-right) to switch basemaps or toggle the overlay.
    """
    if rgb is not None:
        da_ll, bounds = reproject_bands(ds, list(rgb))
        img = _rgba_rgb(da_ll.values, gamma)
        label = f"GOES RGB {tuple(rgb)}"
    else:
        da_ll, bounds = reproject_bands(ds, [band])
        a = da_ll.isel(band=0).values
        lo, hi = _stretch(a, vmin, vmax)
        img = _rgba_single(a, cmap or _cmap(cmi(ds, band)), lo, hi)
        label = f"GOES band {band}"
    (s, w), (n, e) = bounds
    m = folium.Map(location=[(s + n) / 2, (w + e) / 2], zoom_start=5,
                   tiles="OpenStreetMap")
    folium.TileLayer("CartoDB positron", name="light").add_to(m)
    folium.TileLayer(
        tiles=("https://server.arcgisonline.com/ArcGIS/rest/services/"
               "World_Imagery/MapServer/tile/{z}/{y}/{x}"),
        attr="Esri World Imagery", name="satellite",
    ).add_to(m)
    ImageOverlay(_data_uri(img), bounds=bounds, opacity=opacity,
                 name=label).add_to(m)
    folium.LayerControl().add_to(m)
    m.fit_bounds(bounds)
    return m

## 1. Choose a date & file

Each downloaded day has one image (18:00 UTC). If a day has several, `open_goes`
lists them — pass `file_index=` to choose.

In [ ]:
DATE = date(2024, 11, 17)
ds = open_goes(DATE, file_index=0)

## 2. Quick static look

Fast inline render of the full CONUS frame (no map). Reflectance bands are
grayscale; IR bands invert so cold cloud tops are white.

In [ ]:
show_band(ds, 13);   # try 2 (red), 8 (water vapor), 7, ...

In [ ]:
show_rgb(ds, (2, 3, 1));   # true color (daytime)

## 3. Interactive map — scroll to zoom, drag to pan

The image is reprojected to lat/lon and overlaid on a basemap. **Scroll to zoom,
drag to pan**, and use the layer control (top-right) to switch basemap or toggle
the overlay. (First map for a date is a little slow — it reprojects the frame.)

In [ ]:
show_on_map(ds, band=13, opacity=0.85)

In [ ]:
show_on_map(ds, rgb=(2, 3, 1), opacity=0.9)

## 4. Focus a region

Crop first (lon/lat or pixels) to reproject just an area — faster, and the map
opens already zoomed in. Then keep scroll-zooming for detail.

In [ ]:
sub = crop_lonlat(ds, lon_min=-98, lon_max=-92, lat_min=28, lat_max=33)
show_on_map(sub, rgb=(2, 3, 1))

## Tips

- **Change the view:** edit `DATE`, the band in `show_on_map(ds, band=N)`, or the RGB triple.
- **Basemaps:** OSM, light (CartoDB), and satellite (Esri) — switch via the layer control.
- **Opacity:** `show_on_map(ds, band=13, opacity=0.6)` to see the basemap through the imagery.
- **Contrast:** `show_on_map(ds, band=13, vmin=200, vmax=300)` (Kelvin for IR; 0–1 reflectance).
- **Speed:** `crop_lonlat(...)` / `crop_pixels(...)` before mapping to reproject a smaller area.